# Task 2: Colorization of Historical Photographs
### Intership task 2: Faithful replication of historical hues (WWII, 1920s, etc.)

> "Create a model for colorizing historical images, with a focus on faithfully replicating the hues of specific historical periods (for example, World War II and the 1920s). The emphasis should be on historical accuracy in colorization."


In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'

import torch
import numpy as np
from PIL import Image
import cv2
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
import gradio as gr

print(f"--- System Diagnostic (4GB Optimization Mode) ---")
print(f"Torch version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA available (Torch): {cuda_available}")

if cuda_available:
    try:
        print(f"CUDA Device Count: {torch.cuda.device_count()}")
        print(f"Current Device: {torch.cuda.get_device_name(0)}")
    except Exception as e:
        print(f"Warning: CUDA detected but failed to query device details: {e}")
        cuda_available = False

device = "cuda" if cuda_available else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"----------------------------------------------")
print(f"Selected Device: {device} | Dtype: {dtype}")

A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'


--- System Diagnostic (4GB Optimization Mode) ---
Torch version: 2.0.1+cu118
CUDA available (Torch): True
CUDA Device Count: 1
Current Device: NVIDIA GeForce RTX 3050 4GB Laptop GPU
----------------------------------------------
Selected Device: cuda | Dtype: torch.float16


In [ ]:
print(f"Loading Historical Restoration Models with 4GB VRAM Optimizations...")
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_canny", torch_dtype=dtype)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=dtype
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

if device == "cuda":
    # HIGH-SPEED 4GB OPTIMIZATIONS
    pipe.enable_model_cpu_offload() # Loads models one by one into GPU memory
    pipe.enable_vae_slicing()      # Decodes image in slices to save VRAM
    pipe.enable_vae_tiling()       # Allows processing of larger images in tiles
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("xformers optimization: ENABLED")
    except Exception as e:
        print(f"xformers not available, using attention slicing instead.")
        pipe.enable_attention_slicing()
else:
    pipe.to(device)

print("Models loaded and optimized successfully!")

Loading Historical Restoration Models with 4GB VRAM Optimizations...


d:\Downloads\image_gen_colorization\.venv\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
text_encoder\model.safetensors not found


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.


xformers optimization: ENABLED
Models loaded and optimized successfully!


In [ ]:
def restore_photo(image, era):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    # Resize to 512px for model efficiency & memory safety
    image = image.convert("RGB").resize((512, 512))
    
    # Preprocess: Grayscale -> Canny for structure
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    edges = np.stack([edges, edges, edges], axis=2)
    canny_img = Image.fromarray(edges)
    
    # Accurate Historical Styles
    era_styles = {
        "WWII (1940s)": "World War II archival photo, 1940s color film, authentic olive drab military colors, grainy historical look",
        "Roaring 1920s": "1920s vintage colorized look, soft sepia-tinted colors, muted tones, high grain, jazz era aesthetic",
        "Victorian (Colonial)": "Victorian era hand-painted photo, deep rich shadows, ivory skin tones, authentic antique textures",
        "Modern Realism": "high-definition realistic colorization, modern high-fidelity results"
    }
    
    style_prompt = era_styles.get(era, "accurately colorized historical photo")
    full_prompt = f"{style_prompt}, masterpiece, highly detailed, sharp results"
    neg_prompt = "neon, glowing, plastic, cartoon, modern, fake, artistic, painted, oversaturated, unrealistic"
    
    # Inference (10 steps for speed < 20s on 4GB VRAM)
    result = pipe(
        full_prompt, 
        image=canny_img, 
        negative_prompt=neg_prompt, 
        num_inference_steps=10,
        guidance_scale=7.5
    ).images[0]
    
    return result



In [ ]:
# Launch UI
with gr.Blocks(title="Task 2: Accurate Historical Restoration") as demo:
    gr.Markdown("# 📜 Accurate Historical Colorization Engine")
    gr.Markdown("**4GB VRAM Optimization Enabled.** Image processing should now take 10-20 seconds on your RTX 3050.")
    
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(label="Upload or Click Example", type="pil")
            era_opt = gr.Dropdown(
                choices=["WWII (1940s)", "Roaring 1920s", "Victorian (Colonial)", "Modern Realism"], 
                value="WWII (1940s)",
                label="Select Historical Period for Hue Extraction"
            )
            btn = gr.Button("Execute Historical Restoration", variant="primary")
        
        with gr.Column():
            output_img = gr.Image(label="Historically Accurate Result")
    
    # Example photos from folder
    gr.Examples(
        examples=["ww2.jpg", "1920.jpg"],
        inputs=input_img,
        label="Task 2 Datasets (Archive Photos)"
    )
    
    btn.click(fn=restore_photo, inputs=[input_img, era_opt], outputs=output_img)

demo.queue().launch(share=True)


Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://7e8d81f3b9dc91964d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]